# Misure di singoli punti, da confrontare a posteriori

Per quando **a ogni misura cambi qualcosa** (potenza, polarizzazione, filtri, ...) e vuoi confrontare le misure solo dopo.

**Misurare (celle 1-5).** Ogni volta che lanci la cella 5 fai **una misura** e la salvi in `misure_singole/` con un numero progressivo, un `NOME` e delle `NOTE` (scrivi lì cosa hai cambiato, per esempio `potenza 2 mW, polarizzazione 45`). Il registro `misure_singole/misure.csv` tiene l'elenco di tutte le misure (anche di sessioni diverse). Per la misura successiva cambi ciò che devi, modifichi `NOME`/`NOTE` in cella 5 e la rilanci.
- `CON_SCANSIONE = False`: **senza scansione** (laser fermo, scope in single con trigger forzato): ogni misura è un livello di CH2.
- `CON_SCANSIONE = True`: **con la scansione del laser** 1520-1570 nm: ogni misura è uno spettro in funzione della rampa di CH1.
- `POSIZIONE = (X, Y)` in cella 5 muove i motori prima di misurare (serve `USA_MOTORI = True`); `None` = misura dove sei.

**Confrontare (celle 6-8, solo analisi, senza strumenti).** La cella 7 elenca le misure. La cella 8 sovrappone quelle scelte in `SELEZIONE` e calcola il **rapporto di ognuna con la misura di riferimento** `RIFERIMENTO_ID`.

Ordine: celle 1-4 una volta per sessione, poi la cella 5 per ogni misura. Per la sola analisi salta alla cella 6.


In [ ]:
# Cella 1 - configurazione e funzioni degli strumenti
import csv
import os
import sys
import time
import datetime

import numpy as np
import pyvisa

OSC_VISA = "USB::0x0699::0x0408::C028813::INSTR"
OSC_TIMEOUT_MS = 30000
CHANNELS = ["CH1", "CH2"]

CARTELLA_MISURE = "misure_singole"   # cartella dove si accumulano tutte le misure e il registro misure.csv
CON_SCANSIONE = False        # True: scansione del laser (spettro); False: laser fermo, trigger forzato (livello)
USA_MOTORI = False           # True: si collegano i motori e si puo' usare POSIZIONE in cella 5
N_CAMPIONI = 100000          # senza scansione: campioni letti per misura (None = tutto il record, ~240 MB); con scansione si legge sempre tutto
ASSESTAMENTO_S = 1.0         # pausa dopo il movimento dei motori, prima di misurare

LASER_WAVELENGTH_START = 1520.0  # nm, come trigger_scan.m
LASER_WAVELENGTH_STOP = 1570.0   # nm, come trigger_scan.m
ATTESA_RITORNO_LASER_S = 5.0     # dopo uno scan, perche' il laser torni a 1520 nm
RESETTA_CONTATORE_SCAN = True    # prima di ogni start: OUTP:SCAN:STOP + SOUR:WAVE:DESSCANS 1

POLL_INTERVAL_S = 0.1
CHANNEL_READ_PAUSE_S = 0.1


def configura_scope(osc):
    osc.write("HORizontal:MODE MANUAL")
    rec_len = int(osc.query("HORizontal:RECORDLENGTH?"))
    ultimo = rec_len if (CON_SCANSIONE or N_CAMPIONI is None) else min(rec_len, int(N_CAMPIONI))
    osc.write("DATA:START 1")
    osc.write(f"DATA:STOP {ultimo}")
    osc.write("DATA:WIDTH 1")
    osc.write("DATA:ENCdg ASCii")


def arm_acquisition(osc, timeout_s=120):
    osc.write("ACQ:STOPAFTER SEQUENCE")
    osc.write("ACQ:STATE RUN")
    # Lo scan parte solo quando lo scope e' READY (finito il pre-trigger).
    t0 = time.time()
    while osc.query("TRIGger:STATE?").strip().upper() != "READY":
        if time.time() - t0 > timeout_s:
            raise TimeoutError(f"Lo scope non e' arrivato a READY in {timeout_s} s")
        time.sleep(POLL_INTERVAL_S)


def wait_for_acquisition(osc):
    while int(osc.query("ACQ:STATE?")) != 0:
        time.sleep(POLL_INTERVAL_S)


def acquire_channel(osc, source="CH1"):
    osc.write(f"DATA:SOURCE {source}")
    ymult = float(osc.query("WFMPRE:YMULT?"))
    yoff = float(osc.query("WFMPRE:YOFF?"))
    yzero = float(osc.query("WFMPRE:YZERO?"))
    ydata = np.array(osc.query("CURVE?").split(","), dtype=float)
    return (ydata - yoff) * ymult + yzero


In [ ]:
# Cella 2 - connessione all'oscilloscopio (una volta per sessione)
rm = pyvisa.ResourceManager()
osc = rm.open_resource(OSC_VISA)
osc.timeout = OSC_TIMEOUT_MS
configura_scope(osc)
print("Oscilloscopio connesso e configurato.")

In [ ]:
# Cella 3 - laser (serve solo se CON_SCANSIONE = True): carica la DLL e apre il dispositivo
if CON_SCANSIONE:
    USBADDR = 1
    DEVICE_ID = 0x100A  # hex2dec('100A')

    dll_path = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "scripts", "laser", "UsbDllWrap.dll"))
    if not os.path.isfile(dll_path):
        raise RuntimeError(f"UsbDllWrap.dll non trovata in {dll_path} (il notebook va aperto da Chiara_Bru/runs_oscilloscope)")
    print(f"DLL trovata: {dll_path}")
    sys.path.append(os.path.dirname(dll_path))

    import clr
    clr.AddReference("UsbDllWrap")
    import Newport
    from System.Text import StringBuilder

    NP_USB = Newport.USBComm.USB()

    esito = NP_USB.OpenDevices(DEVICE_ID)
    print(f"OpenDevices({hex(DEVICE_ID)}) -> {esito}")
    querydata = StringBuilder(64)


    def laser_query(comando):
        esito = NP_USB.Query(USBADDR, comando, querydata)
        risposta = querydata.ToString()
        print(f"Query {comando!r} -> esito={esito}, risposta={risposta!r}")
        return risposta


    LASER_COMANDI_CON_QUERY = True  # True: come i driver di riferimento (Query + lettura dell'OK)


    def laser_write(comando):
        if LASER_COMANDI_CON_QUERY:
            # Con Write la risposta "OK" del laser resta non letta e si accumula,
            # sfasando le query successive e saturando la coda del laser.
            esito = NP_USB.Query(USBADDR, comando, querydata)
            print(f"Write {comando!r} -> esito={esito}, risposta={querydata.ToString()!r}")
        else:
            esito = NP_USB.Write(USBADDR, comando)
            print(f"Write {comando!r} -> esito={esito}")


    laser_query("*IDN?")
    laser_query("OUTP:STAT?")


In [ ]:
# Cella 4 - motori PI C-863 (servono solo se USA_MOTORI = True): X = ASRL4, Y = ASRL5
if USA_MOTORI:
    MOTORE_X = "ASRL4::INSTR"
    MOTORE_Y = "ASRL5::INSTR"
    PI_BAUD = 19200
    PI_AXIS = "1"
    PI_TIMEOUT_MS = 2000
    PI_TIMEOUT_MOVIMENTO_S = 60
    PI_TOLLERANZA_MM = 0.01  # valore usato nella prova sul laboratorio (i motori arrivano entro circa 10 um)


    def pi_apri(porta):
        inst = rm.open_resource(porta)
        inst.baud_rate = PI_BAUD
        inst.timeout = PI_TIMEOUT_MS
        inst.read_termination = "\n"
        inst.write_termination = "\n"
        return inst


    def pi_query(inst, comando):
        return inst.query(comando).strip()


    def pi_valore(inst, comando):
        """Estrae il numero da risposte tipo '1=+0012.3456'."""
        return float(pi_query(inst, comando).split("=")[-1])


    def pi_limiti(inst):
        return pi_valore(inst, f"TMN? {PI_AXIS}"), pi_valore(inst, f"TMX? {PI_AXIS}")


    def pi_muovi(inst, nome, target):
        """Movimento assoluto in mm, con controllo dei limiti prima di muovere."""
        tmin, tmax = pi_limiti(inst)
        if not (tmin <= target <= tmax):
            raise ValueError(f"Motore {nome}: target {target} mm fuori dai limiti {tmin}..{tmax} mm")
        inst.write(f"MOV {PI_AXIS} {target:.4f}")
        time.sleep(0.1)
        t0 = time.time()
        while True:
            in_target = pi_query(inst, f"ONT? {PI_AXIS}").endswith("1")
            pos = pi_valore(inst, f"POS? {PI_AXIS}")
            if in_target and abs(pos - target) <= PI_TOLLERANZA_MM:
                break
            if time.time() - t0 > PI_TIMEOUT_MOVIMENTO_S:
                raise TimeoutError(f"Motore {nome}: non arrivato a {target} mm (posizione {pos} mm)")
            time.sleep(0.1)
        errore = int(pi_query(inst, "ERR?"))
        if errore != 0:
            raise RuntimeError(f"Motore {nome}: ERR? = {errore} dopo il movimento a {target} mm")
        print(f"Motore {nome}: a {pos:.4f} mm")


    motore_x = pi_apri(MOTORE_X)
    motore_y = pi_apri(MOTORE_Y)

    for nome, inst in (("X", motore_x), ("Y", motore_y)):
        errore_residuo = pi_query(inst, "ERR?")  # legge e azzera un eventuale errore vecchio
        tmin, tmax = pi_limiti(inst)
        print(f"Motore {nome}: {pi_query(inst, '*IDN?')}")
        print(f"   posizione {pi_valore(inst, f'POS? {PI_AXIS}')} mm | limiti {tmin}..{tmax} mm | "
              f"servo {pi_query(inst, f'SVO? {PI_AXIS}')} | referenziato {pi_query(inst, f'FRF? {PI_AXIS}')} | "
              f"errore residuo letto: {errore_residuo}")


In [ ]:
# Cella 5 - UNA MISURA: modifica NOME / NOTE / POSIZIONE qui sotto e lancia la cella
NOME = "prova"                                           # nome breve della misura
NOTE = ""                                                # cosa hai cambiato in questa misura (potenza, polarizzazione, ...)
POSIZIONE = None                                         # (X, Y) in mm: muove i motori prima di misurare; None = non muovere

if POSIZIONE is not None and not USA_MOTORI:
    raise ValueError("POSIZIONE richiede USA_MOTORI = True in cella 1 (e la cella 4 eseguita)")
os.makedirs(CARTELLA_MISURE, exist_ok=True)
registro_misure = os.path.join(CARTELLA_MISURE, "misure.csv")
CAMPI_REGISTRO = ["id", "ora", "nome", "x_mm", "y_mm", "con_scansione", "file", "note"]
if not os.path.isfile(registro_misure):
    with open(registro_misure, "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow(CAMPI_REGISTRO)
with open(registro_misure, newline="", encoding="utf-8") as f:
    id_misura = max([int(r["id"]) for r in csv.DictReader(f)] + [0]) + 1

if POSIZIONE is not None:
    x, y = float(POSIZIONE[0]), float(POSIZIONE[1])
    pi_muovi(motore_x, "X", x)
    pi_muovi(motore_y, "Y", y)
    time.sleep(ASSESTAMENTO_S)
else:
    x = y = ""

try:
    if CON_SCANSIONE:
        laser_write(f"SOUR:WAVE:START {LASER_WAVELENGTH_START}")
        laser_write(f"SOUR:WAVE:STOP {LASER_WAVELENGTH_STOP}")
        if RESETTA_CONTATORE_SCAN:
            laser_write("OUTP:SCAN:STOP")
            laser_write("SOUR:WAVE:DESSCANS 1")
        arm_acquisition(osc)
        laser_write("OUTP:SCAN:START")
    else:
        arm_acquisition(osc)
        osc.write("TRIGger FORCe")                       # nessuna scansione: trigger forzato
    wait_for_acquisition(osc)
    time.sleep(1.0)
    data = {}
    for ch in CHANNELS:
        data[ch] = acquire_channel(osc, ch)
        time.sleep(CHANNEL_READ_PAUSE_S)
except BaseException:
    if CON_SCANSIONE:
        print("\nInterrotto: provo a fermare lo scan del laser.")
        try:
            laser_write("OUTP:SCAN:STOP")
        except Exception as e:
            print(f"Impossibile fermare lo scan: {e}")
    raise

nome_sicuro = "".join(c if c.isalnum() or c in "-_" else "_" for c in NOME)
nome_file = f"m{id_misura:03d}_{nome_sicuro}.csv"
min_len = min(len(v) for v in data.values())
np.savetxt(os.path.join(CARTELLA_MISURE, nome_file), np.column_stack([data[ch][:min_len] for ch in CHANNELS]),
           delimiter=",", header=",".join(CHANNELS), comments="")
with open(registro_misure, "a", newline="", encoding="utf-8") as f:
    csv.writer(f).writerow([id_misura, datetime.datetime.now().isoformat(timespec="seconds"), NOME, x, y,
                            CON_SCANSIONE, nome_file, NOTE])
print(f"Misura {id_misura} salvata: {nome_file} | note: {NOTE!r}")
print(f"  mediana di CH2 = {np.median(data['CH2'][:min_len]):.5g} V")
if CON_SCANSIONE:
    print(f"Il laser torna a {LASER_WAVELENGTH_START} nm: attendi {ATTESA_RITORNO_LASER_S:.0f} s prima della misura successiva")
    time.sleep(ATTESA_RITORNO_LASER_S)


In [ ]:
# Cella 6 - ANALISI (non servono gli strumenti): impostazioni e funzioni
import csv
import os

import numpy as np
import matplotlib.pyplot as plt

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    CARTELLA_MISURE
except NameError:
    CARTELLA_MISURE = "misure_singole"     # stessa cartella della cella 1 (serve se rilanci solo l'analisi in un kernel nuovo)

LIVELLO_BUIO_V = 0.0           # livello di CH2 senza luce, sottratto da tutte le misure
ASSE_X = "tensione_ch1"        # misure con scansione: "tensione_ch1" | "lunghezza_onda" (0-10 V <-> 1520-1570 nm: ipotesi da verificare)
V_0, V_1 = 0.0, 10.0
LAMBDA_0, LAMBDA_1 = 1520.0, 1570.0
BLOCCO = 1000                  # campioni per blocco (per l'errore del livello nelle misure senza scansione)

CANALE = "CH2"
CANALE_RAMPA = "CH1"
V_INIZIO_V = 0.9               # rampa di CH1: inizio quando CH1 supera questo valore
TOLLERANZA_FINE_V = 0.3        # rampa di CH1: fine quando CH1 e' a meno di questo dal suo massimo
N_PUNTI_CURVA = 5000           # ogni spettro e' ridotto a circa questo numero di punti (mediana a blocchi)
N_GRIGLIA = 2000               # punti della griglia comune su cui si confrontano gli spettri
SOGLIA_RIF = 0.02              # dove il riferimento e' sotto questa frazione del suo massimo, il rapporto e' NaN


def leggi_csv(percorso):
    if pd is not None:
        df = pd.read_csv(percorso, dtype=np.float32)
        ch1 = df[CANALE_RAMPA].to_numpy(dtype=float) if CANALE_RAMPA in df.columns else None
        return ch1, df[CANALE].to_numpy(dtype=float)
    dati = np.loadtxt(percorso, delimiter=",", skiprows=1)
    return dati[:, 0].astype(float), dati[:, 1].astype(float)


def rampa_ch1(ch1):
    if ch1 is None:
        return None
    sopra = np.flatnonzero(ch1 > V_INIZIO_V)
    if sopra.size == 0:
        return None
    i0 = int(sopra[0])
    i1 = i0 + int(np.flatnonzero(ch1[i0:] >= ch1.max() - TOLLERANZA_FINE_V)[0])
    return (i0, i1) if i1 > i0 + 10 else None


def curva_su_rampa(ch1, ch2):
    """CH2 nella rampa di CH1 come (v, s): v = tensione di CH1 (retta fittata) al centro di ogni blocco, s = mediana di CH2 nel blocco."""
    rampa = rampa_ch1(ch1)
    if rampa is None:
        return None
    i0, i1 = rampa
    m, c0 = np.polyfit(np.arange(i0, i1 + 1), ch1[i0:i1 + 1], 1)     # CH1 e' quantizzato: si usa la retta fittata
    if m <= 0:
        return None
    n = max(1, (i1 - i0 + 1) // int(N_PUNTI_CURVA))
    k = (i1 - i0 + 1) // n
    s = np.median(ch2[i0:i0 + k * n].reshape(k, n), axis=1)
    return m * (i0 + (np.arange(k) + 0.5) * n - 0.5) + c0, s


def asse_x(v):
    if ASSE_X == "tensione_ch1":
        return v, "CH1 (V)"
    if ASSE_X == "lunghezza_onda":
        return LAMBDA_0 + (v - V_0) / (V_1 - V_0) * (LAMBDA_1 - LAMBDA_0), "lunghezza d'onda (nm)"
    raise ValueError("ASSE_X deve essere 'tensione_ch1' o 'lunghezza_onda'")


def carica_registro():
    with open(os.path.join(CARTELLA_MISURE, "misure.csv"), newline="", encoding="utf-8") as f:
        righe = list(csv.DictReader(f))
    for r in righe:
        r["id"] = int(r["id"])
        r["scansione"] = r["con_scansione"] == "True"
    return righe


def carica_misura(m):
    """Con scansione: {'modo': 'scansione', 'v', 's'}; senza: {'modo': 'livello', 'L', 'err'} (buio sottratto)."""
    ch1, ch2 = leggi_csv(os.path.join(CARTELLA_MISURE, m["file"]))
    if m["scansione"]:
        c = curva_su_rampa(ch1, ch2)
        if c is None:
            raise RuntimeError(f"Misura {m['id']}: segnata con scansione ma senza rampa di CH1")
        return {"modo": "scansione", "v": c[0], "s": c[1] - LIVELLO_BUIO_V}
    k = len(ch2) // BLOCCO
    b = np.median(ch2[:k * BLOCCO].reshape(k, BLOCCO), axis=1) if k > 0 else np.array([np.median(ch2)])
    return {"modo": "livello", "L": float(np.median(ch2)) - LIVELLO_BUIO_V,
            "err": float(b.std(ddof=1) / np.sqrt(len(b))) if len(b) > 1 else 0.0}



In [ ]:
# Cella 7 - elenco delle misure fatte
registro = carica_registro()
print(f"{len(registro)} misure in {os.path.abspath(CARTELLA_MISURE)}\n")
for m in registro:
    pos = f" @ ({m['x_mm']}, {m['y_mm']})" if m["x_mm"] != "" else ""
    print(f"{m['id']:>3} | {m['ora']} | {m['nome']:<20} | {'scansione' if m['scansione'] else 'livello  '}{pos}"
          + (f" | {m['note']}" if m["note"] else ""))


In [ ]:
# Cella 8 - confronto delle misure scelte e rapporto con la misura di riferimento
SELEZIONE = None            # lista di id da confrontare, es. [1, 2, 5]; None = tutte (devono essere dello stesso tipo del riferimento)
RIFERIMENTO_ID = None       # id della misura al denominatore dei rapporti; None = la prima della selezione

registro = carica_registro()
sel = [m for m in registro if SELEZIONE is None or m["id"] in SELEZIONE]
ref = next((m for m in sel if RIFERIMENTO_ID is None or m["id"] == RIFERIMENTO_ID), None)
if ref is None:
    raise ValueError("Nessuna misura selezionata / riferimento non trovato: controlla SELEZIONE e RIFERIMENTO_ID")
modo = "scansione" if ref["scansione"] else "livello"
scartate = [m["id"] for m in sel if m["scansione"] != ref["scansione"]]
if scartate:
    print(f"Salto le misure {scartate}: sono di tipo diverso dal riferimento ({modo})")
sel = [m for m in sel if m["scansione"] == ref["scansione"]]
dati = {m["id"]: carica_misura(m) for m in sel}


def etichetta(m):
    return f"{m['id']} {m['nome']}"


if modo == "livello":
    L = np.array([dati[m["id"]]["L"] for m in sel])
    e = np.array([dati[m["id"]]["err"] for m in sel])
    L_ref, e_ref = dati[ref["id"]]["L"], dati[ref["id"]]["err"]
    R = L / L_ref
    eR = np.abs(R) * np.sqrt((e / np.where(L == 0, np.nan, L)) ** 2 + (e_ref / L_ref) ** 2)
    print(f"Riferimento: misura {ref['id']} ({ref['nome']}), livello {L_ref:.5g} V\n")
    print(f"{'id':>3} | {'nome':<20} | {'livello (V)':>22} | {'rapporto / rif.':>20} | note")
    for m, l_i, e_i, r_i, er_i in zip(sel, L, e, R, eR):
        print(f"{m['id']:>3} | {m['nome']:<20} | {l_i:>12.5g} ± {e_i:<7.2g} | {r_i:>10.4f} ± {er_i:<7.4f} | {m['note']}")
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.5))
    pos_barre = np.arange(len(sel))
    nomi = [etichetta(m) for m in sel]
    a1.bar(pos_barre, L, yerr=e, capsize=3)
    a2.bar(pos_barre, R, yerr=eR, capsize=3)
    for a in (a1, a2):
        a.set_xticks(pos_barre)
        a.set_xticklabels(nomi, rotation=30, ha="right")
    a1.set_ylabel(f"{CANALE} (V)")
    a1.set_title("Livello di ogni misura")
    a2.axhline(1, color="gray", lw=0.8, ls=":")
    a2.set_ylabel(f"rapporto con la misura {ref['id']}")
    a2.set_title("Rapporto con il riferimento")
    plt.tight_layout()
    plt.show()
else:
    v_lo = max(dati[m["id"]]["v"][0] for m in sel)
    v_hi = min(dati[m["id"]]["v"][-1] for m in sel)
    griglia = np.linspace(v_lo, v_hi, int(N_GRIGLIA))
    S = {m["id"]: np.interp(griglia, dati[m["id"]]["v"], dati[m["id"]]["s"]) for m in sel}
    S_ref = S[ref["id"]]
    buono = np.abs(S_ref) > SOGLIA_RIF * np.abs(S_ref).max()
    x_plot, etichetta_x = asse_x(griglia)
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
    for m in sel:
        a1.plot(x_plot, S[m["id"]], lw=1.2 if m["id"] != ref["id"] else 2.2, label=etichetta(m))
        if m["id"] != ref["id"]:
            a2.plot(x_plot[buono], S[m["id"]][buono] / S_ref[buono], lw=1.2, label=f"{m['id']} / {ref['id']}")
    a1.set_ylabel(f"{CANALE} (V)")
    a1.legend(loc="best", fontsize=8)
    a1.set_title(f"Spettri (riferimento = misura {ref['id']}, linea spessa)")
    a2.axhline(1, color="gray", lw=0.8, ls=":")
    a2.set_ylabel(f"rapporto con la misura {ref['id']}")
    a2.set_xlabel(etichetta_x)
    a2.legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.show()
    print(f"Riferimento: misura {ref['id']} ({ref['nome']})\n")
    print(f"{'id':>3} | {'nome':<20} | {'media (V)':>10} | {'rapporto medio / rif.':>22} | note")
    for m in sel:
        media = float(np.mean(S[m["id"]]))
        rap = float(np.mean(S[m["id"]][buono] / S_ref[buono]))
        print(f"{m['id']:>3} | {m['nome']:<20} | {media:>10.5g} | {rap:>22.4f} | {m['note']}")
